In [15]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report, f1_score

In [16]:
train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")

# force clean
train_df = train_df.dropna(subset=['headline', 'category']).reset_index(drop=True)
test_df = test_df.dropna(subset=['headline', 'category']).reset_index(drop=True)

X_train = train_df['headline'].astype(str)
y_train = train_df['category']

X_test = test_df['headline'].astype(str)
y_test = test_df['category']

print(X_train.isna().sum())
print(X_test.isna().sum())

0
0


In [17]:
print(train_df['headline'].isnull().sum())
print(test_df['headline'].isnull().sum())

0
0


In [18]:
vectorizers = {
    "BoW" : CountVectorizer(max_features=5000),
    "TF-IDF": TfidfVectorizer(max_features=5000)
}

In [19]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVM": LinearSVC(),
    "Random Forest": RandomForestClassifier(random_state=42)
}

In [20]:
X_train.isnull().sum()

np.int64(0)

In [ ]:
result = []

for vec_name, vectorizer in vectorizers.items():
    
    #transform text
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    for model_name, model in models.items():

        #train model
        model.fit(X_train_vec, y_train)

        #predict
        y_pred = model.predict(X_test_vec)

        #metrics
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')

        result.append({
            "vectorizers": vec_name,
            "Model": model_name,
            "Accuracy": round(acc, 4),
            "F1 Score": round(f1, 4)
        })

In [ ]:
results_df = pd.DataFrame(result)

results_df = results_df.sort_values(by="Accuracy", ascending=False)

results_df

,vectorizers,Model,Accuracy,F1 Score
6,TF-IDF,Linear SVM,0.9241,0.9174
1,BoW,Logistic Regression,0.9199,0.9126
0,BoW,Naive Bayes,0.9187,0.9184
2,BoW,Linear SVM,0.9157,0.9115
5,TF-IDF,Logistic Regression,0.9129,0.8999
7,TF-IDF,Random Forest,0.9110,0.9001
3,BoW,Random Forest,0.9084,0.8985
4,TF-IDF,Naive Bayes,0.8932,0.8680


In [ ]:
best_row = results_df.iloc[0]

print("Best Combination:")
print(best_row)

Best Combination:
vectorizers        TF-IDF
Model          Linear SVM
Accuracy           0.9241
F1 Score           0.9174
Name: 6, dtype: object


In [ ]:
best_vectorizer_name = best_row['vectorizers']
best_model_name = best_row['Model']

print(best_vectorizer_name)
print(best_model_name)

TF-IDF
Linear SVM


In [ ]:
if best_vectorizer_name == "BoW":
    best_vectorizer = CountVectorizer(max_features=5000)
else:
    best_vectorizer = TfidfVectorizer(max_features=5000)

In [ ]:
if best_model_name == "Naive Bayes":
    best_model = MultinomialNB()

elif best_model_name == "Logistic Regression":
    best_model = LogisticRegression(max_iter=1000)

elif best_model_name == "Linear SVM":
    best_model = LinearSVC()

else:
    best_model = RandomForestClassifier(random_state=42)

In [ ]:
X_train_vec = best_vectorizer.fit_transform(X_train)
X_test_vec = best_vectorizer.transform(X_test)

best_model.fit(X_train_vec, y_train)

final_pred = best_model.predict(X_test_vec)

print("FINAL MODEL ACCURACY:", accuracy_score(y_test, final_pred))
print(classification_report(y_test, final_pred))

FINAL MODEL ACCURACY: 0.9241223103057757
              precision    recall  f1-score   support

        News       0.64      0.28      0.39       276
    Politics       0.94      0.98      0.96      7118
      Sports       0.89      0.81      0.85      1015
  Technology       0.84      0.66      0.74       421

    accuracy                           0.92      8830
   macro avg       0.83      0.68      0.73      8830
weighted avg       0.92      0.92      0.92      8830



In [ ]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(best_model, "../models/best_model.pkl")
joblib.dump(best_vectorizer, "../models/best_vectorizer.pkl")

print("Model saved successfully!")

Model saved successfully!
